# Kryptos Colab AI Workbench

This notebook is generated from the Kryptos repo so Colab can do two jobs in one place:

- run Kryptos benchmark passes on a disposable Colab runtime
- use the `google.colab.ai` workflow from Google's public preview notebook to summarize retained candidates and propose next pruning moves

Primary source notebook: [https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb)

Local reference baseline from `gpu_50sweep_default_baseline.json`:
- Attempts: `97002000`
- Elapsed seconds: `67.271848`
- Attempts/second: `1441940.47407`
- Qualified candidates: `919`
- Hydrated candidates: `64`
- Notable displacement-bearing survivors:
  - `PERIPHERALS` / period `24` / hint `155` / delta `-9`
  - `BUILDINGS` / period `7` / hint `181` / delta `-9`
  - `PROTECTION` / period `21` / hint `155` / delta `-9`

Local snapshot workflow:

- the generator also creates a local snapshot archive named `colab_repo_snapshot_posix.zip`
- if GitHub is behind your working tree, upload that archive with the snapshot cell before running the GPU benchmarks

Notes:

- the current repo GPU path is OpenCL-based, so this notebook probes OpenCL explicitly before launching the heavier run
- if the Colab runtime does not expose a working OpenCL platform, use the CPU fallback cell instead of forcing the GPU runner
- keep this work aligned to public, reproducible K4 method reconstruction only


In [ ]:
# @title Configure the workbench
REPO_URL = 'https://github.com/claudlos/Kryptos.git'
REPO_REF = 'main'
WORKDIR = '/content/Kryptos'
BOOTSTRAP_COMMAND = ['-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools>=69', 'wheel']
GPU_INSTALL_COMMAND = ['-m', 'pip', 'install', 'numpy', 'pyopencl']

SMOKE_COMMAND = ['-m', 'kryptos.benchmark_cli', '--runner', 'gpu-opencl', '--profile', 'smoke', '--json', '--output', 'runs/colab_gpu_smoke.json', '--sweeps-per-pass', '1']
HEAVY_COMMAND = ['-m', 'kryptos.benchmark_cli', '--runner', 'gpu-opencl', '--profile', 'default', '--json', '--output', 'runs/colab_gpu_default_50sweeps.json', '--sweeps-per-pass', '50']
CPU_FALLBACK_COMMAND = ['-m', 'kryptos.benchmark_cli', '--runner', 'cpu-strategy', '--profile', 'default', '--json', '--output', 'runs/colab_cpu_default.json', '--dataset-profile', 'full-public', '--scorer-profile', 'anchor-first']

SMOKE_OUTPUT = 'runs/colab_gpu_smoke.json'
HEAVY_OUTPUT = 'runs/colab_gpu_default_50sweeps.json'
CPU_FALLBACK_OUTPUT = 'runs/colab_cpu_default.json'
AI_MODEL_NAME = 'google/gemini-2.0-flash'
LOCAL_SNAPSHOT_NAME = 'colab_repo_snapshot_posix.zip'


In [ ]:
# @title Helper: run commands and show stdout/stderr on failure
import os
import pathlib
import subprocess

def ensure_content_cwd():
    os.chdir("/content")

def resolve_workdir():
    workdir = pathlib.Path(WORKDIR)
    return workdir if workdir.is_absolute() else pathlib.Path("/content") / workdir

def resolve_artifact(path_value):
    candidate = pathlib.Path(path_value)
    return candidate if candidate.is_absolute() else resolve_workdir() / candidate

def run_logged(command, *, cwd=None):
    workdir = pathlib.Path(cwd) if cwd is not None else resolve_workdir()
    if not workdir.exists():
        workdir = pathlib.Path("/content")
        workdir.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env["PYTHONPATH"] = str(workdir) if not env.get("PYTHONPATH") else f"{workdir}:{env['PYTHONPATH']}"
    completed = subprocess.run(
        command,
        cwd=workdir,
        env=env,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()
    return completed


In [ ]:
# @title Clone the repo and prepare the Python environment
import shutil
import sys

for dependency in ("os", "pathlib", "subprocess"):
    if dependency not in globals():
        raise RuntimeError("Run the helper cell before bootstrapping the repo.")

ensure_content_cwd()
workdir = resolve_workdir()
if workdir.exists():
    shutil.rmtree(workdir)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(workdir)],
    check=True,
)
os.chdir(workdir)
command = [sys.executable, *BOOTSTRAP_COMMAND]
print("Launching:", " ".join(command))
run_logged(command, cwd=workdir)
if str(workdir) not in sys.path:
    sys.path.insert(0, str(workdir))
existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    str(workdir) if not existing_pythonpath else f"{workdir}:{existing_pythonpath}"
)
print(f"Bootstrapped {workdir} and exported it to PYTHONPATH")
if not (workdir / "kryptos").exists():
    print("The cloned checkout does not contain the local kryptos package changes.")
    print(f"Upload {LOCAL_SNAPSHOT_NAME} with the snapshot cell below if GitHub is behind your local working tree.")


In [ ]:
# @title Optional: upload a local repo snapshot zip when GitHub is behind
from google.colab import files
import os
import shutil
import sys
import zipfile

ensure_content_cwd()
print(f"Choose {LOCAL_SNAPSHOT_NAME} from your local machine if you need your current unpushed code.")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No snapshot archive was uploaded.")

archive_name, archive_bytes = next(iter(uploaded.items()))
archive_path = pathlib.Path("/content") / archive_name
archive_path.write_bytes(archive_bytes)

workdir = resolve_workdir()
if workdir.exists():
    shutil.rmtree(workdir)
workdir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive_path) as zf:
    zf.extractall(workdir)

if not (workdir / "kryptos").exists():
    raise RuntimeError(
        "Snapshot extracted, but the kryptos package directory is missing. "
        "Regenerate the archive with POSIX-style zip paths and retry."
    )

os.chdir(workdir)
if str(workdir) not in sys.path:
    sys.path.insert(0, str(workdir))
existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    str(workdir) if not existing_pythonpath else f"{workdir}:{existing_pythonpath}"
)
print(f"Extracted snapshot to {workdir} and exported it to PYTHONPATH")


In [ ]:
# @title Install OpenCL diagnostics (safe to rerun)
import subprocess

ensure_content_cwd()
subprocess.run(["apt-get", "update"], check=True)
subprocess.run(["apt-get", "install", "-y", "clinfo", "ocl-icd-opencl-dev", "opencl-headers"], check=True)
subprocess.run(["bash", "-lc", "clinfo | head -n 120"], check=False)


In [ ]:
# @title Install GPU Python packages after OpenCL headers
import pathlib
import sys

ensure_content_cwd()
command = [sys.executable, *GPU_INSTALL_COMMAND]
print("Launching:", " ".join(command))
run_logged(command, cwd=pathlib.Path("/content"))


In [ ]:
# @title Probe OpenCL availability
import json
import pyopencl as cl

platform_summaries = []
for platform in cl.get_platforms():
    platform_summaries.append(
        {
            "name": platform.name,
            "vendor": platform.vendor,
            "version": platform.version,
            "devices": [device.name for device in platform.get_devices()],
        }
    )
print(json.dumps(platform_summaries, indent=2))
if not platform_summaries:
    raise RuntimeError("No OpenCL platforms detected. Use the CPU fallback command in this notebook instead.")


In [ ]:
# @title Run a smoke benchmark first
import sys

ensure_content_cwd()
workdir = resolve_workdir()
if not workdir.exists():
    raise FileNotFoundError(f"{workdir} is missing. Run the clone or snapshot upload cell first.")
command = [sys.executable, *SMOKE_COMMAND]
print("Launching:", " ".join(command))
run_logged(command, cwd=workdir)


In [ ]:
# @title Run the heavier benchmark
import sys
import time

ensure_content_cwd()
workdir = resolve_workdir()
if not workdir.exists():
    raise FileNotFoundError(f"{workdir} is missing. Run the clone or snapshot upload cell first.")
started = time.time()
command = [sys.executable, *HEAVY_COMMAND]
print("Launching:", " ".join(command))
run_logged(command, cwd=workdir)
print(f"Completed in {time.time() - started:.2f}s")


In [ ]:
# @title CPU fallback if OpenCL is unavailable
import sys
import time

ensure_content_cwd()
workdir = resolve_workdir()
if not workdir.exists():
    raise FileNotFoundError(f"{workdir} is missing. Run the clone or snapshot upload cell first.")
started = time.time()
command = [sys.executable, *CPU_FALLBACK_COMMAND]
print("Launching:", " ".join(command))
run_logged(command, cwd=workdir)
print(f"Completed in {time.time() - started:.2f}s")


In [ ]:
# @title Inspect retained candidates
import json

artifact_path = resolve_artifact(HEAVY_OUTPUT)
if not artifact_path.exists():
    artifact_path = resolve_artifact(CPU_FALLBACK_OUTPUT)
payload = json.loads(artifact_path.read_text(encoding="utf-8"))
top_candidates = payload.get("artifacts", {}).get("top_candidates", [])[:10]
for index, candidate in enumerate(top_candidates, start=1):
    summary = {
        "rank": index,
        "keyword": candidate.get("keyword"),
        "period": candidate.get("period"),
        "best_mode": candidate.get("best_mode"),
        "best_score": candidate.get("best_score"),
        "raw_displacement_hint": candidate.get("raw_displacement_hint"),
        "raw_best_displacement": candidate.get("raw_best_displacement"),
        "preview": candidate.get("best_preview"),
    }
    print(json.dumps(summary, indent=2))


In [ ]:
# @title Summarize the retained candidates with Colab AI
from google.colab import ai
import json

artifact_path = resolve_artifact(HEAVY_OUTPUT)
if not artifact_path.exists():
    artifact_path = resolve_artifact(CPU_FALLBACK_OUTPUT)
payload = json.loads(artifact_path.read_text(encoding="utf-8"))
top_candidates = payload.get("artifacts", {}).get("top_candidates", [])[:10]
candidate_json = json.dumps(top_candidates, indent=2)
prompt = (
    "You are analyzing Kryptos K4 benchmark output.\n\n"
    "Constraints:\n"
    "- work only from public, reproducible evidence\n"
    "- do not invent private plaintext, leaked keys, or auction-only material\n"
    "- optimize for the next concrete pruning or search move\n\n"
    "Tasks:\n"
    "1. Summarize the strongest repeated structural signals across the retained candidates.\n"
    "2. Call out any displacement clusters or periodic/transposition fingerprints worth exploiting on GPU.\n"
    "3. Recommend the next benchmark override to try and why.\n\n"
    "Candidate JSON:\n"
    f"{candidate_json}\n"
)
response = ai.generate_text(prompt, model_name=AI_MODEL_NAME)
print(response)


In [ ]:
# Optional: export artifacts to Google Drive
from google.colab import drive
import shutil

drive.mount('/content/drive')
export_dir = pathlib.Path('/content/drive/MyDrive/Kryptos_runs')
export_dir.mkdir(parents=True, exist_ok=True)
for artifact in [SMOKE_OUTPUT, HEAVY_OUTPUT, CPU_FALLBACK_OUTPUT]:
    source = resolve_artifact(artifact)
    if source.exists():
        shutil.copy2(source, export_dir / source.name)
print(f"Copied artifacts to {export_dir}")
